In [1]:
import itertools
import time
import warnings

import numpy as np
import pandas as pd

from sklearn.metrics import roc_auc_score
from sklearn.ensemble import (
    RandomForestClassifier,
    ExtraTreesClassifier,
    GradientBoostingClassifier,
)
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier

warnings.filterwarnings("ignore")

SEED = 20260916

import os
DATA_DIR = next(p for p in [".", "..", "../.."] if os.path.exists(f"{p}/train.csv"))
train = pd.read_csv(f"{DATA_DIR}/train.csv")
validation = pd.read_csv(f"{DATA_DIR}/validation.csv")
test = pd.read_csv(f"{DATA_DIR}/test.csv")
sample_submission = pd.read_csv(f"{DATA_DIR}/sample_submission.csv")

feature_columns = [
    c for c in train.columns
    if c not in ["row_id", "target"]
]

y_train = train["target"].to_numpy()
y_validation = validation["target"].to_numpy()

X_train_raw = train[feature_columns].to_numpy()
X_validation_raw = validation[feature_columns].to_numpy()
X_test_raw = test[feature_columns].to_numpy()


def add_pair_features(X):
    A = X[:, :256]
    B = X[:, 256:]

    abs_diff = np.abs(A - B)

    return np.hstack([
        X,
        abs_diff,
    ])


X_train_pair = add_pair_features(X_train_raw)
X_validation_pair = add_pair_features(X_validation_raw)
X_test_pair = add_pair_features(X_test_raw)

feature_sets = {
    "raw": (
        X_train_raw,
        X_validation_raw,
        X_test_raw,
    ),
    "pair": (
        X_train_pair,
        X_validation_pair,
        X_test_pair,
    ),
}

print("Raw:", X_train_raw.shape)
print("Pair:", X_train_pair.shape)

Raw: (2520, 512)
Pair: (2520, 768)


In [ ]:
model_specs = [
    # Extra Trees
    {
        "family": "Extra Trees",
        "name": "Extra Trees raw",
        "features": "raw",
        "model": ExtraTreesClassifier(
            n_estimators=100,
            max_depth=None,
            max_features=1.0,
            min_samples_leaf=2,
            criterion="gini",
            random_state=SEED,
            n_jobs=-1,
        ),
    },
    {
        "family": "Extra Trees",
        "name": "Extra Trees pair",
        "features": "pair",
        "model": ExtraTreesClassifier(
            n_estimators=100,
            max_depth=None,
            max_features=1.0,
            min_samples_leaf=2,
            criterion="gini",
            random_state=SEED,
            n_jobs=-1,
        ),
    },

    # Logistic Regression
    {
        "family": "Logistic Regression",
        "name": "Logistic raw",
        "features": "raw",
        "model": Pipeline([
            ("scaler", StandardScaler()),
            ("model", LogisticRegression(
                C=0.003,
                solver="liblinear",
                max_iter=5000,
                random_state=SEED,
            )),
        ]),
    },
    {
        "family": "Logistic Regression",
        "name": "Logistic pair",
        "features": "pair",
        "model": Pipeline([
            ("scaler", StandardScaler()),
            ("model", LogisticRegression(
                C=0.003,
                solver="liblinear",
                max_iter=5000,
                random_state=SEED,
            )),
        ]),
    },

    # LightGBM 
    {
        "family": "LightGBM",
        "name": "LightGBM raw",
        "features": "raw",
        "model": LGBMClassifier(
            boosting_type="goss",
            colsample_bytree=0.7,
            learning_rate=0.05,
            max_depth=5,
            min_child_samples=20,
            n_estimators=500,
            num_leaves=31,
            reg_lambda=0.05,
            random_state=SEED,
            n_jobs=-1,
            verbosity=-1,
        ),
    },
    {
        "family": "LightGBM",
        "name": "LightGBM pair",
        "features": "pair",
        "model": LGBMClassifier(
            boosting_type="goss",
            colsample_bytree=0.7,
            learning_rate=0.05,
            max_depth=5,
            min_child_samples=20,
            n_estimators=500,
            num_leaves=31,
            reg_lambda=0.05,
            random_state=SEED,
            n_jobs=-1,
            verbosity=-1,
        ),
    },

    # XGBoost
    {
        "family": "XGBoost",
        "name": "XGBoost raw",
        "features": "raw",
        "model": XGBClassifier(
            n_estimators=100,
            max_depth=5,
            learning_rate=0.4,
            subsample=1.0,
            colsample_bytree=1.0,
            min_child_weight=1,
            random_state=SEED,
            n_jobs=-1,
            eval_metric="auc",
        ),
    },
    {
        "family": "XGBoost",
        "name": "XGBoost pair",
        "features": "pair",
        "model": XGBClassifier(
            n_estimators=100,
            max_depth=5,
            learning_rate=0.4,
            subsample=1.0,
            colsample_bytree=1.0,
            min_child_weight=1,
            random_state=SEED,
            n_jobs=-1,
            eval_metric="auc",
        ),
    },

    # CatBoost
    {
        "family": "CatBoost",
        "name": "CatBoost raw",
        "features": "raw",
        "model": CatBoostClassifier(
            iterations=2000,
            depth=6,
            learning_rate=0.05,
            l2_leaf_reg=3,
            random_strength=2,
            random_seed=SEED,
            verbose=False,
            allow_writing_files=False,
        ),
    },
    {
        "family": "CatBoost",
        "name": "CatBoost pair",
        "features": "pair",
        "model": CatBoostClassifier(
            iterations=2000,
            depth=6,
            learning_rate=0.05,
            l2_leaf_reg=3,
            random_strength=2,
            random_seed=SEED,
            verbose=False,
            allow_writing_files=False,
        ),
    },

    # RF
    {
        "family": "Random Forest",
        "name": "Random Forest",
        "features": "raw",
        "model": RandomForestClassifier(
            n_estimators=500,
            max_depth=None,
            max_features=1.0,
            min_samples_leaf=2,
            criterion="gini",
            random_state=SEED,
            n_jobs=-1,
        ),
    },

    # Gradient Boosting
    {
        "family": "Gradient Boosting",
        "name": "Gradient Boosting",
        "features": "raw",
        "model": GradientBoostingClassifier(
            n_estimators=100,
            learning_rate=0.1,
            max_depth=3,
            random_state=SEED,
        ),
    },
]

In [3]:
validation_predictions = {}
test_predictions = {}

individual_results = []

for i, spec in enumerate(model_specs, start=1):
    name = spec["name"]
    feature_name = spec["features"]
    model = spec["model"]

    X_train, X_validation, X_test = feature_sets[
        feature_name
    ]

    print(f"[{i}/{len(model_specs)}] {name}")

    start = time.perf_counter()

    model.fit(
        X_train,
        y_train,
    )

    val_pred = model.predict_proba(
        X_validation
    )[:, 1]

    test_pred = model.predict_proba(
        X_test
    )[:, 1]

    auc = roc_auc_score(
        y_validation,
        val_pred,
    )

    elapsed = time.perf_counter() - start

    validation_predictions[name] = val_pred
    test_predictions[name] = test_pred

    individual_results.append({
        "family": spec["family"],
        "model": name,
        "roc_auc": auc,
        "time_sec": elapsed,
    })

individual_df = (
    pd.DataFrame(individual_results)
    .sort_values("roc_auc", ascending=False)
    .reset_index(drop=True)
)

individual_df

[1/12] Extra Trees raw
[2/12] Extra Trees pair
[3/12] Logistic raw
[4/12] Logistic pair
[5/12] LightGBM raw
[6/12] LightGBM pair
[7/12] XGBoost raw
[8/12] XGBoost pair
[9/12] CatBoost raw
[10/12] CatBoost pair
[11/12] Random Forest
[12/12] Gradient Boosting


,family,model,roc_auc,time_sec
0,Extra Trees,Extra Trees pair,0.924643,0.760923
1,CatBoost,CatBoost pair,0.921148,20.402672
2,LightGBM,LightGBM raw,0.920753,1.947866
3,XGBoost,XGBoost raw,0.920153,0.522924
4,Extra Trees,Extra Trees raw,0.919994,0.552489
5,CatBoost,CatBoost raw,0.919821,14.363525
6,LightGBM,LightGBM pair,0.919783,2.175249
7,Logistic Regression,Logistic pair,0.916250,0.107576
8,Logistic Regression,Logistic raw,0.915676,0.068663
9,Random Forest,Random Forest,0.913431,14.947421


In [4]:
best_by_family = (
    individual_df
    .sort_values("roc_auc", ascending=False)
    .groupby("family", as_index=False)
    .first()
    .sort_values("roc_auc", ascending=False)
    .reset_index(drop=True)
)

best_by_family

,family,model,roc_auc,time_sec
0,Extra Trees,Extra Trees pair,0.924643,0.760923
1,CatBoost,CatBoost pair,0.921148,20.402672
2,LightGBM,LightGBM raw,0.920753,1.947866
3,XGBoost,XGBoost raw,0.920153,0.522924
4,Logistic Regression,Logistic pair,0.916250,0.107576
5,Random Forest,Random Forest,0.913431,14.947421
6,Gradient Boosting,Gradient Boosting,0.905051,15.629896


In [5]:
strong_models = (
    best_by_family[
        best_by_family["roc_auc"] >= 0.91
    ]["model"]
    .tolist()
)

print("Strong models:")
for model in strong_models:
    print(model)

Strong models:
Extra Trees pair
CatBoost pair
LightGBM raw
XGBoost raw
Logistic pair
Random Forest


In [6]:
def rank_predictions(predictions):
    return (
        pd.Series(predictions)
        .rank(method="average", pct=True)
        .to_numpy()
    )

In [7]:
blend_results = []

for size in range(2, len(strong_models) + 1):

    for combination in itertools.combinations(
        strong_models,
        size,
    ):
        # Probability average
        probability_blend = np.mean(
            [
                validation_predictions[name]
                for name in combination
            ],
            axis=0,
        )

        probability_auc = roc_auc_score(
            y_validation,
            probability_blend,
        )

        blend_results.append({
            "method": "probability",
            "models": combination,
            "n_models": size,
            "roc_auc": probability_auc,
        })

        # Rank average
        rank_blend = np.mean(
            [
                rank_predictions(
                    validation_predictions[name]
                )
                for name in combination
            ],
            axis=0,
        )

        rank_auc = roc_auc_score(
            y_validation,
            rank_blend,
        )

        blend_results.append({
            "method": "rank",
            "models": combination,
            "n_models": size,
            "roc_auc": rank_auc,
        })


blend_df = (
    pd.DataFrame(blend_results)
    .sort_values("roc_auc", ascending=False)
    .reset_index(drop=True)
)

blend_df.head(20)

,method,models,n_models,roc_auc
0,rank,"(Extra Trees pair, XGBoost raw)",2,0.932596
1,rank,"(Extra Trees pair, XGBoost raw, Logistic pair)",3,0.932500
2,rank,"(Extra Trees pair, LightGBM raw, XGBoost raw, ...",4,0.932054
3,rank,"(Extra Trees pair, CatBoost pair, XGBoost raw,...",4,0.931703
4,probability,"(Extra Trees pair, XGBoost raw, Logistic pair)",3,0.931620
5,rank,"(Extra Trees pair, LightGBM raw, XGBoost raw)",3,0.931422
6,probability,"(Extra Trees pair, XGBoost raw, Logistic pair,...",4,0.931378
7,probability,"(Extra Trees pair, LightGBM raw, XGBoost raw, ...",4,0.931288
8,rank,"(Extra Trees pair, CatBoost pair, LightGBM raw...",5,0.931059
9,rank,"(Extra Trees pair, CatBoost pair, LightGBM raw...",4,0.930931


In [8]:
best_blend = blend_df.iloc[0]

print("Best method:", best_blend["method"])
print("Best models:")

for model in best_blend["models"]:
    print("-", model)

print("Validation ROC-AUC:", best_blend["roc_auc"])

Best method: rank
Best models:
- Extra Trees pair
- XGBoost raw
Validation ROC-AUC: 0.9325956632653061


In [9]:
best_models = list(best_blend["models"])
best_method = best_blend["method"]

if best_method == "probability":

    final_test_prediction = np.mean(
        [
            test_predictions[name]
            for name in best_models
        ],
        axis=0,
    )

else:

    final_test_prediction = np.mean(
        [
            rank_predictions(
                test_predictions[name]
            )
            for name in best_models
        ],
        axis=0,
    )

In [10]:
submission = sample_submission.copy()

submission["target"] = final_test_prediction

submission.to_csv(
    "../submission_final_blend.csv",
    index=False,
)

submission.head()

,row_id,target
0,row_46ad1afad8562239f79c,0.823925
1,row_b42ade9cd57b8d919acb,0.769041
2,row_c951cd5e197b6b4e8b83,0.445565
3,row_a54b3423760d599416ce,0.096326
4,row_a87c87cc47690e2a303f,0.864247


In [11]:
print(submission.shape)
print(submission["target"].min())
print(submission["target"].max())
print(submission["target"].isna().sum())

(1116, 2)
0.0035842293906810036
0.9793906810035842
0


In [12]:
individual_df

,family,model,roc_auc,time_sec
0,Extra Trees,Extra Trees pair,0.924643,0.760923
1,CatBoost,CatBoost pair,0.921148,20.402672
2,LightGBM,LightGBM raw,0.920753,1.947866
3,XGBoost,XGBoost raw,0.920153,0.522924
4,Extra Trees,Extra Trees raw,0.919994,0.552489
5,CatBoost,CatBoost raw,0.919821,14.363525
6,LightGBM,LightGBM pair,0.919783,2.175249
7,Logistic Regression,Logistic pair,0.916250,0.107576
8,Logistic Regression,Logistic raw,0.915676,0.068663
9,Random Forest,Random Forest,0.913431,14.947421


In [13]:
blend_df.head(20)

,method,models,n_models,roc_auc
0,rank,"(Extra Trees pair, XGBoost raw)",2,0.932596
1,rank,"(Extra Trees pair, XGBoost raw, Logistic pair)",3,0.932500
2,rank,"(Extra Trees pair, LightGBM raw, XGBoost raw, ...",4,0.932054
3,rank,"(Extra Trees pair, CatBoost pair, XGBoost raw,...",4,0.931703
4,probability,"(Extra Trees pair, XGBoost raw, Logistic pair)",3,0.931620
5,rank,"(Extra Trees pair, LightGBM raw, XGBoost raw)",3,0.931422
6,probability,"(Extra Trees pair, XGBoost raw, Logistic pair,...",4,0.931378
7,probability,"(Extra Trees pair, LightGBM raw, XGBoost raw, ...",4,0.931288
8,rank,"(Extra Trees pair, CatBoost pair, LightGBM raw...",5,0.931059
9,rank,"(Extra Trees pair, CatBoost pair, LightGBM raw...",4,0.930931


In [14]:
extra_test_rank = (
    pd.Series(test_predictions["Extra Trees pair"])
    .rank(method="average", pct=True)
    .to_numpy()
)

xgboost_test_rank = (
    pd.Series(test_predictions["XGBoost raw"])
    .rank(method="average", pct=True)
    .to_numpy()
)

final_test_prediction = (
    0.5 * extra_test_rank
    + 0.5 * xgboost_test_rank
)

submission = sample_submission.copy()
submission["target"] = final_test_prediction

submission.to_csv(
    "../submission_rank_extra_xgb.csv",
    index=False,
)

print(submission.head())
print("Rows:", len(submission))
print("NaN:", submission["target"].isna().sum())
print("Min:", submission["target"].min())
print("Max:", submission["target"].max())

                     row_id    target
0  row_46ad1afad8562239f79c  0.823925
1  row_b42ade9cd57b8d919acb  0.769041
2  row_c951cd5e197b6b4e8b83  0.445565
3  row_a54b3423760d599416ce  0.096326
4  row_a87c87cc47690e2a303f  0.864247
Rows: 1116
NaN: 0
Min: 0.0035842293906810036
Max: 0.9793906810035842
